# Análisis Exploratorio de Datos: Uso del Suelo a lo largo del Tiempo (2011, 2014, 2021)

Este notebook analiza la evolución del uso del suelo y el Índice Ambiental Ponderado (WEI) para los 21 municipios de Cundinamarca en los años 2011, 2014 y 2021.

Utilizamos las matrices de transición de uso del suelo para los periods 2011-2014, 2014-2017 y 2018-2021 para calcular:
- Área por clase de uso del suelo (Forest formation, Agricultural and livestock area, Non-vegetated area, Water body)
- Porcentajes de cada clase
- Índice Ambiental Ponderado (WEI)

Luego, creamos gráficos interactivos de líneas con Plotly donde:
- Eje X: años (2011, 2014, 2021)
- Eje Y: valor (área, porcentaje o WEI)
- Cada municipio es una línea (diferenciada por color)
- Cada clase es un tipo de línea (continuo, punteado, etc.)
- Incluimos casillas de verificación para mostrar/ocultar municipios y clases


In [6]:
# Importaciones y configuración inicial
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import sys
import re
import unicodedata

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from ipywidgets import Checkbox, VBox, HBox, Label, Layout, Dropdown, Output
from IPython.display import display, clear_output

# Añadir el directorio raíz al path para importar módulos locales
cwd = Path.cwd().resolve()
root = cwd.parent if cwd.name == 'src' else cwd
sys.path.append(str(root / 'src'))

# Importar funciones auxiliares del análisis existente
from soil_poverty_analysis import (
    find_repo_root,
    parse_weight_series,
    normalize_municipality,
)

# Encontrar la raíz del repositorio
root = find_repo_root(root)
DATA_DIR = root / 'data'
TRANSITIONS_DIR = DATA_DIR / 'transitions'

# Configuración de estilo para gráficos
import plotly.io as pio
pio.templates.default = "simple_white"

In [7]:
# Funciones auxiliares (copiadas del notebook 30_transicion_pobreza_dispersion_adj.ipynb)
manual_mapping = {
    'bojaca': 'Bojacá',
    'cajica': 'Cajicá',
    'chia': 'Chía',
    'facatativa': 'Facatativá',
    'fusagasuga': 'Fusagasugá',
    'gachancipa': 'Gachancipá',
    'sibate': 'Sibaté',
    'sopo': 'Sopó',
    'tocancipa': 'Tocancipá',
    'zipacon': 'Zipacón',
    'zipaquira': 'Zipaquirá',
    'rosal': 'el rosal',
    'calera': 'la calera',
}

def load_semicolon_csv(path, columns=None, encodings=('utf-8', 'latin1')):
    last_error = None
    for encoding in encodings:
        try:
            if columns is None:
                return pd.read_csv(path, sep=';', encoding=encoding)
            return pd.read_csv(path, sep=';', encoding=encoding, usecols=list(columns))
        except Exception as exc:
            last_error = exc
    raise last_error

def load_transition(path):
    df = pd.read_csv(path, sep=',')
    df.columns = ['from_class', 'to_class', 'area_ha']
    df['area_ha'] = pd.to_numeric(df['area_ha'], errors='coerce')
    return df

def compute_transition_indices(path, municipio_base):
    df = load_transition(path)
    total = df['area_ha'].sum()
    f = df.set_index(['from_class', 'to_class'])['area_ha'].to_dict()
    get = lambda a, b: f.get((a, b), 0.0)
    F_ForAgro = get('Forest formation', 'Agricultural and livestock area')
    F_WatAgro = get('Water body', 'Agricultural and livestock area')
    F_AgroFor = get('Agricultural and livestock area', 'Forest formation')
    F_AgroWat = get('Agricultural and livestock area', 'Water body')
    F_ForFor = get('Forest formation', 'Forest formation')
    F_WatWat = get('Water body', 'Water body')
    F_AgroAgro = get('Agricultural and livestock area', 'Agricultural and livestock area')
    A_total = total
    degrad_total = (F_ForAgro + F_WatAgro) / A_total * 100 if A_total else np.nan
    recover_total = (F_AgroFor + F_AgroWat) / A_total * 100 if A_total else np.nan
    degrad_forest_rel = F_ForAgro / F_ForFor if F_ForFor else np.nan
    degrad_water_rel = F_WatAgro / F_WatWat if F_WatWat else np.nan
    recover_agro_rel = F_AgroFor / F_AgroAgro if F_AgroAgro else np.nan
    ind_neto_transicion = degrad_total - recover_total
    ind_neto_transicion_rel = (degrad_forest_rel + degrad_water_rel) - recover_agro_rel
    return {
        'municipio_base': municipio_base,
        'A_total': A_total,
        'F_ForAgro': F_ForAgro,
        'F_WatAgro': F_WatAgro,
        'F_AgroFor': F_AgroFor,
        'F_AgroWat': F_AgroWat,
        'F_ForFor': F_ForFor,
        'F_WatWat': F_WatWat,
        'F_AgroAgro': F_AgroAgro,
        'degrad_total': degrad_total,
        'recover_total': recover_total,
        'degrad_forest_rel': degrad_forest_rel,
        'degrad_water_rel': degrad_water_rel,
        'recover_agro_rel': recover_agro_rel,
        'ind_neto_transicion': ind_neto_transicion,
        'ind_neto_transicion_rel': ind_neto_transicion_rel,
    }


In [8]:
# Definir los períodos de transición y los años de interés
transition_periods = [
    (2011, 2014, 'Transition_*_2011-2014.csv'),
    (2014, 2017, 'Transition_*_2014-2017.csv'),
    (2018, 2021, 'Transition_*_2018-2021.csv'),
]

# Diccionarios para almacenar los resultados por año
data_2011 = {}
data_2014 = {}
data_2021 = {}

# Procesar cada período de transición
for start_year, end_year, pattern in transition_periods:
    print(f"Procesando período {start_year}-{end_year}...")
    transition_files = list(TRANSITIONS_DIR.glob(pattern))
    print(f"  Encontrados {len(transition_files)} archivos")
    
    for path in transition_files:
        # Extraer el nombre del municipio del archivo
        raw_name = path.name.replace('Transition_', '').replace(f'_{start_year}-{end_year}.csv', '')
        normalized = normalize_municipality(raw_name)
        municipio_base = manual_mapping.get(normalized, normalized)
        
        # Cargar la transición
        df_trans = load_transition(path)
        
        # Calcular área por clase para el año de inicio (from_class)
        area_from = df_trans.groupby('from_class')['area_ha'].sum()
        # Calcular área por clase para el año de fin (to_class)
        area_to = df_trans.groupby('to_class')['area_ha'].sum()
        # Área total (debe ser la misma para ambos años asumiendo conservación de área total)
        total_area = df_trans['area_ha'].sum()
        
        # Clases de interés
        classes_of_interest = [
            'Forest formation',
            'Agricultural and livestock area',
            'Non-vegetated area',
            'Water body'
        ]
        
        # Procesar según el año
        if start_year == 2011 and end_year == 2014:
            # Para 2011: usar área de inicio (from_class)
            if municipio_base not in data_2011:
                data_2011[municipio_base] = {'total_area': total_area}
            for cls in classes_of_interest:
                data_2011[municipio_base][cls] = area_from.get(cls, 0.0)
            # Para 2014: usar área de fin (to_class) del mismo período
            if municipio_base not in data_2014:
                data_2014[municipio_base] = {'total_area': total_area}
            for cls in classes_of_interest:
                data_2014[municipio_base][cls] = area_to.get(cls, 0.0)
        elif start_year == 2014 and end_year == 2017:
            # Para 2014: también podemos usar el año de inicio de 2014-2017 (debería ser consistente con el anterior)
            # Pero ya lo tenemos de 2011-2014, así que lo verificamos
            if municipio_base not in data_2014:
                data_2014[municipio_base] = {'total_area': total_area}
            for cls in classes_of_interest:
                # Si ya existe, promediamos? Pero asumimos consistencia.
                # Si no existe, lo establecemos.
                if cls not in data_2014[municipio_base]:
                    data_2014[municipio_base][cls] = area_from.get(cls, 0.0)
            # Para 2017: no lo necesitamos
        elif start_year == 2018 and end_year == 2021:
            # Para 2021: usar área de fin (to_class)
            if municipio_base not in data_2021:
                data_2021[municipio_base] = {'total_area': total_area}
            for cls in classes_of_interest:
                data_2021[municipio_base][cls] = area_to.get(cls, 0.0)
        else:
            print(f"    Advertencia: período {start_year}-{end_year} no procesado para años de interés")
    print(f"  Procesado {len(transition_files)} archivos para {start_year}-{end_year}")


Procesando período 2011-2014...
  Encontrados 22 archivos
  Procesado 22 archivos para 2011-2014
Procesando período 2014-2017...
  Encontrados 22 archivos
  Procesado 22 archivos para 2014-2017
Procesando período 2018-2021...
  Encontrados 22 archivos
  Procesado 22 archivos para 2018-2021


In [9]:
# Convertir los diccionarios a DataFrames y calcular porcentajes y WEI
def compute_percentages_and_wei(data_dict, year_label):
    """
    Convierte el diccionario de datos para un año en un DataFrame y calcula porcentajes y WEI.
    """
    records = []
    for municipio, values in data_dict.items():
        total_area = values['total_area']
        record = {'municipio_base': municipio, 'year': year_label, 'total_area': total_area}
        
        classes = ['Forest formation', 'Agricultural and livestock area', 'Non-vegetated area', 'Water body']
        areas = {}
        for cls in classes:
            area_val = values.get(cls, 0.0)
            areas[cls] = area_val
            record[f'{cls}_area'] = area_val
        
        # Calcular porcentajes
        for cls in classes:
            area_val = areas[cls]
            pct = (area_val / total_area * 100) if total_area > 0 else 0.0
            record[f'{cls}_perc'] = pct
        
        # Calcular WEI
        wei_weights = {
            'Forest formation': 1.0,
            'Agricultural and livestock area': 0.5,
            'Non-vegetated area': 0.0,
            'Water body': 1.0
        }
        wei = sum(areas[cls] * wei_weights[cls] for cls in classes) / total_area if total_area > 0 else 0.0
        record['WEI'] = wei
        
        records.append(record)
    
    df = pd.DataFrame(records)
    return df

# Procesar cada año
df_2011 = compute_percentages_and_wei(data_2011, 2011)
df_2014 = compute_percentages_and_wei(data_2014, 2014)
df_2021 = compute_percentages_and_wei(data_2021, 2021)

# Combinar todos los años
df_all = pd.concat([df_2011, df_2014, df_2021], ignore_index=True)

# Ordenar por municipio y año
df_all = df_all.sort_values(['municipio_base', 'year']).reset_index(drop=True)

# Mostrar una vista previa
print("DataFrame combinado (primeras 10 filas):")
display(df_all.head(10))

# Verificar que tenemos datos para los 21 municipios en cada año
print("\nConteo de municipios por año:")
print(df_all['year'].value_counts().sort_index())

DataFrame combinado (primeras 10 filas):


,municipio_base,year,total_area,Forest formation_area,Agricultural and livestock area_area,Non-vegetated area_area,Water body_area,Forest formation_perc,Agricultural and livestock area_perc,Non-vegetated area_perc,Water body_perc,WEI
0,Bojacá,2011,10231.053234,1694.900465,6521.247594,351.344646,12.296135,16.566236,63.739748,3.434100,0.120184,0.485563
1,Bojacá,2014,10231.053234,1634.485514,6451.213696,381.284885,11.939562,15.975731,63.055226,3.726741,0.116699,0.476200
2,Bojacá,2021,10231.053234,1616.397393,6527.582081,512.986977,8.999197,15.798934,63.801663,5.014019,0.087960,0.477877
3,Cajicá,2011,5131.741013,366.254465,3890.951583,791.340062,20.309459,7.137041,75.821277,15.420499,0.395762,0.454434
4,Cajicá,2014,5131.741013,338.196701,3846.589715,838.729741,34.472337,6.590292,74.956817,16.343961,0.671747,0.447404
5,Cajicá,2021,5131.741013,360.387121,3509.421439,1181.499268,19.240115,7.022707,68.386566,23.023361,0.374924,0.415909
6,Chía,2011,7992.731674,1471.743570,5028.658990,1226.772690,11.670121,18.413524,62.915399,15.348603,0.146009,0.500172
7,Chía,2014,7992.731674,1460.607514,4958.817849,1296.168966,25.656566,18.274197,62.041590,16.216846,0.320999,0.496160
8,Chía,2021,7992.731674,1454.639737,4609.964761,1612.952146,13.362679,18.199532,57.676961,20.180236,0.167185,0.472052
9,Facatativá,2011,15804.407016,2401.136059,12147.883507,1041.564535,26.905853,15.192826,76.863899,6.590342,0.170243,0.537950



Conteo de municipios por año:
year
2011    22
2014    22
2021    22
Name: count, dtype: int64


In [10]:
class_cols = {
    'Forest formation': 'Forest formation_perc',
    'Agricultural and livestock area': 'Agricultural and livestock area_perc',
    'Non-vegetated area': 'Non-vegetated area_perc',
    'Water body': 'Water body_perc',
    'WEI': 'WEI',
}

df_plot = df_all[['municipio_base', 'year'] + list(class_cols.values())].rename(
    columns={v: k for k, v in class_cols.items()}
)

df_long = df_plot.melt(
    id_vars=['municipio_base', 'year'],
    var_name='class',
    value_name='value'
)

fig = go.Figure()

for (municipio, cls), group in df_long.groupby(['municipio_base', 'class']):
    fig.add_trace(
        go.Scatter(
            x=group['year'],
            y=group['value'],
            mode='lines+markers',
            name=f'{municipio} - {cls}',
            legendgroup=municipio,
            line=dict(
                dash=line_styles.get(cls, 'solid'),
                width=2,
            ),
            marker=dict(size=5),
            hovertemplate='<b>%{fullData.name}</b><br>Año: %{x}<br>Valor: %{y:.2f}<extra></extra>'
        )
    )

fig.update_layout(
    title='Evolución del uso del suelo y WEI por municipio',
    xaxis=dict(title='Año', dtick=1),
    yaxis=dict(title='Porcentaje / WEI'),
    legend=dict(traceorder='grouped', font=dict(size=9)),
    hovermode='closest',
    template='simple_white'
)

fig